In [1]:
import sys
from pathlib import Path

notebook_cwd = Path.cwd().resolve()
project_root = (
    notebook_cwd.parent
    if notebook_cwd.name == "discovery"
    else notebook_cwd
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import asyncio
import random
import sqlite3
import textwrap
from contextlib import closing
from typing import Any

import pandas as pd
from pydantic import BaseModel, ConfigDict, Field

from synth_extract.agents.llm import (  # noqa: E402
    LLMBackend,
    StructuredAgent,
    StructuredSuccess,
    StructuredTask,
)

In [3]:
discovery_db_path = project_root / "discovery" / "discovery_workspace.db"
fulltext_root = project_root / "data" / "fulltext"

print(f"Database: {discovery_db_path}")
print(f"Full text: {fulltext_root}")

Database: /Users/kevinge/Work/Data Extraction/synth_extract/discovery/discovery_workspace.db
Full text: /Users/kevinge/Work/Data Extraction/synth_extract/data/fulltext


In [4]:
class ExtractionModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Sample(ExtractionModel):
    name: str = Field(
        description="Polymer or material name as written in the paper, not a code."
    )
    identifier: str = Field(
        description="Label the paper uses for this specific sample; empty if none."
    )
    aliases: list[str] = Field(
        default_factory=list,
        description="Other names, codes, or identifiers used for this same sample.",
    )
    description: str = Field(
        description="2-4 sentence description using only information from the paper."
    )


class SampleList(ExtractionModel):
    samples: list[Sample] = Field(default_factory=list)


SampleList.model_validate({"samples": []})

SampleList(samples=[])

In [ ]:
SAMPLE_PROMPT = """
Identify every distinct polymer material sample prepared in this paper and describe each one.

A sample is a polymer or polymer-based material made in this work. Samples are
defined by synthesis: if the synthesis or formulation procedure differs (monomers,
monomer ratio, composition, filler or additive loading, molecular weight, amounts or reaction
conditions such as temperature, time, catalyst, or solvent), it is a separate sample,
even if the paper only reports it in a table or figure. Do not merge samples that
differ in synthesis. When in doubt, list it as its own sample.
A chemical modification of a sample produces a new sample.
Test or measurement conditions do not create new samples.

DO NOT INCLUDE AS SAMPLES
- commercial materials used as received, only as constituents of another sample;
- individual monomers, initiators, catalysts, solvents, crosslinkers, additives,
  fillers, drugs, or other components when they are not themselves the sample;
- characterization instruments or measurement substrates;
- computational models;
- materials reported only from other papers;
- intermediates that are only used to prepare another sample (their synthesis
  belongs to that sample).

DESCRIPTION
The description should contain enough information to understand and distinguish
the sample without reading the surrounding paper.
Include, when reported:
1. what the material is: polymer chemistry, composition, architecture, or form;
2. how it was made: the key synthesis steps and conditions needed to identify it;
3. what distinguishes its synthesis from the other samples in the paper;
4. explicit values when they distinguish samples ("PLLA block Mn 20 kg/mol", not
   "longer PLLA block").

Every statement must come from the paper. Do not infer, generalize, or add background
knowledge. If something is not mentioned in the paper, it does not go in the description.

FIELDS
- "name": the polymer or material name as written in the paper (e.g. "PEG-b-PLLA",
  "PVDF membrane"). Not a code or label. Different samples may share a name.
- "identifier": the label the paper uses for this specific sample (e.g. "PEL-10",
  "Sample 7", "MN3"); empty string if none
- "aliases": every other name, code, or identifier the paper uses for this same sample
- "description"

OUTPUT
Return exactly:

{
  "samples": [
    {
      "name": "",
      "identifier": "",
      "aliases": [],
      "description": ""
    }
  ]
}

If no qualifying samples are present, return:

{"samples": []}

Return only the structured output.

"""

USER_TEMPLATE = """
<paper>
{markdown_full_text}
</paper>
"""

In [6]:
host = "localhost"
port = "8000"
base_url = f"http://{host}:{port}/v1"

model = "qwen3.6-27b"
api_key = "none"

max_tokens = 150000
extra_body = {"chat_template_kwargs": {"enable_thinking": True}}

backend = LLMBackend(
    model=model,
    base_url=base_url,
    api_key=api_key,
    temperature=0.0,
    timeout=600,
    max_tokens=max_tokens,
    extra_body=extra_body,
)
agent = StructuredAgent(backend)

In [7]:
sample_list_response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "sample_list",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "samples": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "identifier": {"type": "string"},
                            "aliases": {
                                "type": "array",
                                "items": {"type": "string"},
                            },
                            "description": {"type": "string"},
                        },
                        "required": ["name", "identifier", "aliases", "description"],
                        "additionalProperties": False,
                    },
                }
            },
            "required": ["samples"],
            "additionalProperties": False,
        },
    },
}


class SampleExtractionRequest(ExtractionModel):
    uid: str
    markdown_full_text: str


def build_sample_extraction_prompt(request: SampleExtractionRequest) -> str:
    return USER_TEMPLATE.format(markdown_full_text=request.markdown_full_text)


sample_extraction_task = StructuredTask[SampleExtractionRequest, SampleList](
    name="sample_extraction",
    prompt_builder=build_sample_extraction_prompt,
    output_model=SampleList,
    response_format=sample_list_response_format,
    system_prompt=SAMPLE_PROMPT.strip(),
)

In [8]:
# Read paper metadata without modifying the discovery database.
database_uri = f"{discovery_db_path.as_uri()}?mode=ro"
with closing(sqlite3.connect(database_uri, uri=True, timeout=60)) as conn:
    conn.execute("PRAGMA query_only = ON")
    papers_df = pd.read_sql_query(
        """
        SELECT
            paper_uid AS uid,
            canonical_source AS source,
            title,
            category_class AS category,
            fulltext_path
        FROM polymer_material_papers
        ORDER BY paper_uid
        """,
        conn,
    )


def resolve_markdown_path(row: pd.Series) -> Path:
    """Resolve the local Markdown corresponding to a database paper."""
    stored_value = row["fulltext_path"]
    candidates: list[Path] = []
    if isinstance(stored_value, str) and stored_value.strip():
        stored_path = Path(stored_value.strip())
        if not stored_path.is_absolute():
            stored_path = project_root / stored_path
        candidates.append(stored_path.with_suffix(".md"))
    candidates.append(
        fulltext_root / str(row["source"]) / str(row["uid"]) / f"{row['uid']}.md"
    )
    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return candidate.resolve()
    return candidates[0].resolve()


papers_df["paper_path"] = papers_df.apply(resolve_markdown_path, axis=1)
papers_df = papers_df.loc[papers_df["paper_path"].map(Path.is_file)].reset_index(drop=True)
print(f"Papers with Markdown: {len(papers_df):,}")

Papers with Markdown: 180


In [9]:
# Papers checked by hand, with the expected sample count (borderline cases in brackets).
expected_samples = {
    "ID000968724": "1",
    "ID000798019": "1 (+~7)",
    "ID000873705": "2",
    "ID000785400": "3",
    "ID001073028": "4 (+1)",
    "ID000754133": "4 (+2)",
    "ID000312202": "19",
    "ID000931235": "4",
    "ID000744781": "9",
    "ID000711118": "2",
    "ID000760683": "2",
    "ID000593638": "6",
    "ID000137616": "4",
    "ID000573326": "6",
    "ID001100444": "3",
    "ID000006174": "~20",
    "ID000162162": "3",
    "ID000147139": "2",
    "ID000108553": "3 (+9)",
    "ID000246349": "5",
}

# Use the hand-checked papers, or set random_n to sample other papers instead.
random_n: int | None = None
random_seed = 7

if random_n is None:
    selected_df = papers_df.loc[papers_df["uid"].isin(expected_samples)]
else:
    selected_df = papers_df.sample(n=random_n, random_state=random_seed)

selected_paper_records: list[dict[str, Any]] = selected_df.to_dict(orient="records")
max_parallel_requests = 8
print(f"Papers selected: {len(selected_paper_records):,}")

Papers selected: 20


In [10]:
async def extract_paper(
    number: int,
    paper: dict[str, Any],
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    uid = str(paper["uid"])
    total = len(selected_paper_records)
    try:
        async with semaphore:
            markdown_full_text = await asyncio.to_thread(
                Path(paper["paper_path"]).read_text,
                encoding="utf-8",
                errors="replace",
            )
            request = SampleExtractionRequest(
                uid=uid,
                markdown_full_text=markdown_full_text,
            )
            outcome = await agent.arun(sample_extraction_task, request)

        if not isinstance(outcome, StructuredSuccess):
            print(f"[{number}/{total}] UID: {uid} | {outcome.error_type}: {outcome.message}")
            return {
                **paper,
                "status": "error",
                "result": None,
                "raw_response": outcome.raw_response,
                "error_type": outcome.error_type,
                "error_message": outcome.message,
            }

        print(f"[{number}/{total}] UID: {uid} | samples: {len(outcome.result.samples)}")
        return {
            **paper,
            "status": "ok",
            "result": outcome.result,
            "raw_response": outcome.raw_response,
            "error_type": None,
            "error_message": None,
        }
    except Exception as exc:
        print(f"[{number}/{total}] UID: {uid} | {type(exc).__name__}: {exc}")
        return {
            **paper,
            "status": "error",
            "result": None,
            "raw_response": None,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }


async def extract_selected_papers(
    papers: list[dict[str, Any]],
    max_parallel: int = 8,
) -> list[dict[str, Any]]:
    semaphore = asyncio.Semaphore(max_parallel)
    tasks = [
        asyncio.create_task(extract_paper(number, paper, semaphore))
        for number, paper in enumerate(papers, start=1)
    ]
    return await asyncio.gather(*tasks)

In [11]:
extraction_runs = await extract_selected_papers(
    selected_paper_records,
    max_parallel=max_parallel_requests,
)

sample_review_df = pd.DataFrame(
    {
        "uid": run["uid"],
        "category": run["category"],
        "title": run["title"],
        "status": run["status"],
        "n_samples": len(run["result"].samples) if run["result"] is not None else None,
        "expected": expected_samples.get(run["uid"]),
        "samples": run["result"].model_dump()["samples"] if run["result"] is not None else None,
        "error_type": run["error_type"],
        "error_message": run["error_message"],
    }
    for run in extraction_runs
)

print(f"Successful: {(sample_review_df['status'] == 'ok').sum():,}")
print(f"Failed: {(sample_review_df['status'] != 'ok').sum():,}")
print(f"Total samples: {int(sample_review_df['n_samples'].fillna(0).sum()):,}")
display(sample_review_df[["uid", "title", "status", "n_samples", "expected"]])

[5/20] UID: ID000162162 | samples: 3
[3/20] UID: ID000137616 | samples: 4
[8/20] UID: ID000573326 | samples: 6
[6/20] UID: ID000246349 | samples: 4
[2/20] UID: ID000108553 | samples: 2
[4/20] UID: ID000147139 | samples: 3
[10/20] UID: ID000711118 | samples: 2
[7/20] UID: ID000312202 | samples: 19
[9/20] UID: ID000593638 | samples: 5
[13/20] UID: ID000760683 | samples: 2
[14/20] UID: ID000785400 | samples: 3
[18/20] UID: ID000968724 | samples: 1
[1/20] UID: ID000006174 | samples: 27
[16/20] UID: ID000873705 | samples: 2
[12/20] UID: ID000754133 | samples: 5
[17/20] UID: ID000931235 | samples: 4
[15/20] UID: ID000798019 | samples: 3
[11/20] UID: ID000744781 | samples: 12
[19/20] UID: ID001073028 | samples: 4
[20/20] UID: ID001100444 | samples: 5
Successful: 20
Failed: 0
Total samples: 116


,uid,title,status,n_samples,expected
0,ID000006174,Incorporation of adhesion peptides into nonadh...,ok,27,~20
1,ID000108553,Polyol and polyurethane foam from cellulose hy...,ok,2,3 (+9)
2,ID000137616,Experimental and numerical investigation of me...,ok,4,4
3,ID000147139,In-situ preparation and electrochromic propert...,ok,3,2
4,ID000162162,The Regulation of Local Li+ Coordination Envir...,ok,3,3
5,ID000246349,Bioresponsive and transformable coacervate act...,ok,4,5
6,ID000312202,Activated polyallene as versatile substrate fo...,ok,19,19
7,ID000573326,"Microstructure regulation, thermal degradation...",ok,6,6
8,ID000593638,Fluorescent zinc acrylate resins containing co...,ok,5,6
9,ID000711118,Immiscible Polymer Blends Made from Industrial...,ok,2,2


In [12]:
WIDTH = 160

for _, row in sample_review_df.iterrows():
    print("=" * WIDTH)
    print(f"{row['uid']} | {row['title']}")
    print(f"samples: {row['n_samples']} | expected: {row['expected']}")
    if row["status"] != "ok":
        print(f"ERROR {row['error_type']}: {row['error_message']}")
        continue
    for i, sample in enumerate(row["samples"], start=1):
        print("-" * WIDTH)
        print(f"[{i}] name: {sample['name']}")
        print(f"    identifier: {sample['identifier']}")
        print(f"    aliases: {', '.join(sample['aliases'])}")
        print(textwrap.fill(
            sample["description"],
            width=WIDTH,
            initial_indent="    ",
            subsequent_indent="    ",
        ))

ID000006174 | Incorporation of adhesion peptides into nonadhesive hydrogels useful for tissue resurfacing
samples: 27 | expected: ~20
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[1] name: PEG diacrylate (MW 8000)
    identifier: 
    aliases: PEG 8000 diacrylate, PEG diacrylate prepolymer
    A diacrylated poly(ethylene glycol) prepolymer synthesized by reacting PEG (MW 8000) with acryloyl chloride. PEG was dried by azeotropic distillation in
    benzene, then reacted with triethylamine (2 mol per PEG hydroxyl end group) and acryloyl chloride (3 mol per PEG hydroxyl end group) under argon reflux for
    4 h. The mixture was cooled overnight, filtered, distilled to half volume, precipitated in hexane, and dried in vacuo.
---------------------------------------------------------------------------------------------------------------------------------------------------------

In [25]:
samples_df = pd.DataFrame(
    {
        "uid": row["uid"],
        "title": row["title"],
        **sample,
    }
    for _, row in sample_review_df.iterrows()
    if row["samples"] is not None
    for sample in row["samples"]
)
display(samples_df)

,uid,title,name,identifier,aliases,description
0,ID000006174,Incorporation of adhesion peptides into nonadh...,PEG diacrylate (MW 8000),,"[PEG 8000 diacrylate, PEG diacrylate prepolymer]",Diacrylated poly(ethylene glycol) prepolymer s...
1,ID000006174,Incorporation of adhesion peptides into nonadh...,PEG homonetwork hydrogel,,[10% PEG diacrylate hydrogel],Crosslinked PEG diacrylate hydrogel formed by ...
2,ID000006174,Incorporation of adhesion peptides into nonadh...,PEG homonetwork hydrogel,,[23% PEG diacrylate hydrogel],Crosslinked PEG diacrylate hydrogel formed by ...
3,ID000006174,Incorporation of adhesion peptides into nonadh...,Copolymer hydrogel,,[23% PEG diacrylate + acrylated YRGDS (no spac...,Copolymer hydrogel formed by adding lyophilize...
4,ID000006174,Incorporation of adhesion peptides into nonadh...,Copolymer hydrogel,,[23% PEG diacrylate + acrylated YRGDS (no spac...,Copolymer hydrogel formed by adding lyophilize...
...,...,...,...,...,...,...
137,ID001100444,3D Printable Plasmonic Titanium Nitride Nanopa...,Neat thermoplastic polyurethane,N-TPU,"[0 wt% T-TPU, T-TPU with 0 wt% TiN]",Thermoplastic polyurethane (TPU) composite fil...
138,ID001100444,3D Printable Plasmonic Titanium Nitride Nanopa...,T-TPU composite filament,0.5 wt% T-TPU,[T-TPU with 0.5 wt% TiN],Thermoplastic polyurethane (TPU) composite fil...
139,ID001100444,3D Printable Plasmonic Titanium Nitride Nanopa...,T-TPU composite filament,1 wt% T-TPU,[T-TPU with 1 wt% TiN],Thermoplastic polyurethane (TPU) composite fil...
140,ID001100444,3D Printable Plasmonic Titanium Nitride Nanopa...,T-TPU composite filament,2 wt% T-TPU,[T-TPU with 2 wt% TiN],Thermoplastic polyurethane (TPU) composite fil...


In [29]:
# Add sample storage and per-paper extraction progress to polymer_samples.db.
import sqlite3

samples_db_path = project_root / "data" / "polymer_samples.db"
if not samples_db_path.is_file():
    raise FileNotFoundError(f"Database not found: {samples_db_path}")

with sqlite3.connect(samples_db_path) as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS samples (
            sample_id INTEGER PRIMARY KEY AUTOINCREMENT,
            sample_uid TEXT GENERATED ALWAYS AS (
                'SID' || printf('%06d', sample_id)
            ) STORED UNIQUE,
            paper_uid TEXT NOT NULL
                REFERENCES papers(paper_uid),
            name TEXT NOT NULL,
            identifier TEXT NOT NULL DEFAULT '',
            aliases TEXT NOT NULL DEFAULT '[]'
                CHECK (json_valid(aliases)),
            description TEXT NOT NULL
        );

        CREATE INDEX IF NOT EXISTS idx_samples_paper_uid
            ON samples(paper_uid);

        CREATE TABLE IF NOT EXISTS sample_extraction_progress (
            paper_uid TEXT PRIMARY KEY
                REFERENCES papers(paper_uid),
            status TEXT NOT NULL DEFAULT 'pending'
                CHECK (status IN ('pending', 'in_progress', 'success', 'failed')),
            attempt_count INTEGER NOT NULL DEFAULT 0
                CHECK (attempt_count >= 0),
            last_error TEXT,
            started_at TEXT,
            completed_at TEXT,
            updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
        );

        CREATE INDEX IF NOT EXISTS idx_sample_extraction_progress_status
            ON sample_extraction_progress(status);

        INSERT OR IGNORE INTO sample_extraction_progress (paper_uid)
        SELECT paper_uid FROM papers;
        """
    )

    paper_count = conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
    progress_count = conn.execute("SELECT COUNT(*) FROM sample_extraction_progress").fetchone()[0]
    sample_count = conn.execute("SELECT COUNT(*) FROM samples").fetchone()[0]

print(f"Database: {samples_db_path}")
print(f"samples rows: {sample_count:,}")
print(f"sample_extraction_progress rows: {progress_count:,} / papers: {paper_count:,}")


Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/polymer_samples.db
samples rows: 0
sample_extraction_progress rows: 273,191 / papers: 273,191


In [13]:
# Create sample-extraction UID manifests from pending progress rows.
samples_db_path = project_root / "data" / "polymer_samples.db"
uid_manifest_dir = project_root / "data" / "uid_manifest"
manifest_prefix = "sample_extraction_uids_"
uids_per_file = 3_000

if not samples_db_path.is_file():
    raise FileNotFoundError(f"Database not found: {samples_db_path}")
database_uri = f"{samples_db_path.as_uri()}?mode=ro"
with closing(sqlite3.connect(database_uri, uri=True, timeout=60)) as conn:
    conn.execute("PRAGMA query_only = ON")
    conn.execute("PRAGMA busy_timeout = 60000")
    pending_uids = [
        str(row[0])
        for row in conn.execute(
            """
            SELECT paper_uid
            FROM sample_extraction_progress
            WHERE status IS NULL OR status = 'pending'
            ORDER BY paper_uid
            """
        )
    ]

manifest_uids = pending_uids
if not manifest_uids:
    raise RuntimeError("No pending sample-extraction UIDs were found")
if len(manifest_uids) != len(set(manifest_uids)):
    raise RuntimeError("Pending sample-extraction UIDs are not unique")

uid_manifest_dir.mkdir(parents=True, exist_ok=True)
for temporary_path in uid_manifest_dir.glob(f"{manifest_prefix}*.txt.tmp"):
    temporary_path.unlink()

manifest_rows = []
staged_files = []
for start in range(0, len(manifest_uids), uids_per_file):
    chunk = manifest_uids[start : start + uids_per_file]
    file_number = len(staged_files) + 1
    final_path = uid_manifest_dir / f"{manifest_prefix}{file_number:04d}.txt"
    temporary_path = final_path.with_suffix(".txt.tmp")
    temporary_path.write_text("\n".join(chunk) + "\n", encoding="utf-8")
    staged_files.append((temporary_path, final_path))
    manifest_rows.append(
        {
            "file": final_path.name,
            "uids": len(chunk),
            "first_uid": chunk[0],
            "last_uid": chunk[-1],
        }
    )

written_uid_count = sum(row["uids"] for row in manifest_rows)
if written_uid_count != len(manifest_uids):
    raise RuntimeError("The staged manifest UID count is incorrect")
if any(row["uids"] > uids_per_file for row in manifest_rows):
    raise RuntimeError("A manifest exceeds the 3,000-UID limit")

# Replace only manifests owned by this cell after staging succeeds.
for old_path in uid_manifest_dir.glob(f"{manifest_prefix}*.txt"):
    old_path.unlink()
for temporary_path, final_path in staged_files:
    temporary_path.replace(final_path)

sample_manifest_summary = pd.DataFrame(manifest_rows)
print(f"Manifest directory: {uid_manifest_dir}")
print(f"Pending progress rows: {len(pending_uids):,}")
print(f"UIDs written: {len(manifest_uids):,}")
print(f"Manifest files: {len(staged_files):,}")
print(f"Maximum UIDs per file: {uids_per_file:,}")
sample_manifest_summary

Manifest directory: /Users/kevinge/Work/Data Extraction/synth_extract/data/uid_manifest
Pending progress rows: 273,191
UIDs written: 273,191
Manifest files: 92
Maximum UIDs per file: 3,000


,file,uids,first_uid,last_uid
0,sample_extraction_uids_0001.txt,3000,ID000000608,ID000005905
1,sample_extraction_uids_0002.txt,3000,ID000005906,ID000011088
2,sample_extraction_uids_0003.txt,3000,ID000011089,ID000027040
3,sample_extraction_uids_0004.txt,3000,ID000027042,ID000033604
4,sample_extraction_uids_0005.txt,3000,ID000033608,ID000041529
...,...,...,...,...
87,sample_extraction_uids_0088.txt,3000,ID001031771,ID001037960
88,sample_extraction_uids_0089.txt,3000,ID001037965,ID001044028
89,sample_extraction_uids_0090.txt,3000,ID001044029,ID001072084
90,sample_extraction_uids_0091.txt,3000,ID001072085,ID001099985
